# Fase 8: Youtube Transcript API

Este notebook implementa um teste para a YoutubeTranscriptAPI

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
import pandas as pd
import time
import random
import os
from tqdm import tqdm

In [6]:
caminho_arquivo = "../data/preprocessed_title_description.csv"
df = pd.read_csv(caminho_arquivo)

print(len(df))

if 'transcription' not in df.columns:
    df['transcription'] = None

9270


In [7]:
api = YouTubeTranscriptApi()

In [8]:
def transcrever_video(id_video):
    try:
        # Tenta buscar a legenda em pt, pt-BR ou pt-PT
        transcrito = api.fetch(video_id=id_video, languages=['pt', 'pt-BR', 'pt-PT'])
        
        texto_completo = " ".join([trecho.text for trecho in transcrito.snippets])
        return texto_completo, False
        
    except Exception as e:
        erro_str = str(e).lower()
        # Verifica se o YouTube bloqueou o IP
        if "too many requests" in erro_str or "cookie" in erro_str or "blocked" in erro_str:
            print(f"\n[ALERTA] BAN DETECTADO NO IP! Vídeo {id_video}")
            return None, True 
        
        # Se for outro erro (ex: vídeo deletado, sem legenda, etc), apenas retorna vazio 
        return None, False

In [9]:
def extrair_transcricoes_df(df, caminho_salvamento, limite_videos=500):
    print(f"\nIniciando coleta... Limite desta rodada: {limite_videos} vídeos.\n")
    
    videos_processados = 0
    
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Baixando Transcrições"):
        
        # Trava de segurança para não rodar infinitamente
        if videos_processados >= limite_videos:
            print(f"\nLimite de {limite_videos} vídeos atingido para esta rodada.")
            break
            
        # Pula se a linha já tiver uma transcrição salva 
        if pd.notna(row['transcription']) and row['transcription'] != "":
            continue
            
        id_video = row['videoId']
        
        transcricao, ip_block = transcrever_video(id_video)
        
        if ip_block:
            print("\nExecução abortada por bloqueio de IP. Volte amanhã ou troque de rede.")
            break
            
        df.at[index, 'transcription'] = transcricao
        videos_processados += 1
        
        if videos_processados % 50 == 0:
            df.to_csv(caminho_salvamento, index=False)
            
        time.sleep(1 + random.random())

    # Salva o arquivo final da rodada
    df.to_csv(caminho_salvamento, index=False)
    print(f"\nRodada finalizada! Progresso salvo em {caminho_salvamento}")
    
    return df

In [10]:
df = extrair_transcricoes_df(df, caminho_arquivo, limite_videos=100)


Iniciando coleta... Limite desta rodada: 100 vídeos.



Baixando Transcrições:   0%|          | 0/9270 [00:00<?, ?it/s]

/tmp/ipykernel_12312/2616621862.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Eu sou polícia, mano. Não vou ser polícia e vou fingir de morto, né, pô. Mano, tô sentindo que vamos meter a faca em mim, mano. Epa. Ah, eu vou ter que fazer meu papel de polícia. Cadê? Quem é? Quem é? Quem é? É a careca. Sai do meio, careca. >> Ave Maria. >> Já toma, né? E toma o quê? Toma um emotezinho aqui, ó. Toma o emotezinho aí, ó. Ó. Vai tomando aí do Felc, ó. Ó, ó a quebradinha. Isso aí não, hein, Fel.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'transcription'] = transcricao
Baixando Transcrições:   0%|          | 22/9270 [00:55<6:26:05,  2.50s/it]


[ALERTA] BAN DETECTADO NO IP! Vídeo NAjhZCyUKsM

Execução abortada por bloqueio de IP. Volte amanhã ou troque de rede.



Rodada finalizada! Progresso salvo em ../data/preprocessed_title_description.csv


In [12]:
caminho_arquivo = "../data/preprocessed_title_description.csv"
df = pd.read_csv(caminho_arquivo)

df_amostra = df[df['transcription'].notna() & (df['transcription'] != "")].copy()

colunas_analise = ['videoId', 'title', 'transcription']
df_amostra = df_amostra[colunas_analise]

print(f"Total de vídeos na amostra: {len(df_amostra)}")

caminho_amostra = "../data/amostra_comparacao_transcricao.csv"
df_amostra.to_csv(caminho_amostra, index=False)

print(f"Arquivo salvo em: {caminho_amostra}")
display(df_amostra.head())

Total de vídeos na amostra: 19
Arquivo salvo em: ../data/amostra_comparacao_transcricao.csv


,videoId,title,transcription
0,w22lrhGXy_I,FELCA O MELHOR XERIFE DO MM2! #roblox,"Eu sou polícia, mano. Não vou ser polícia e vo..."
1,wpRkmL6Hbbo,Dep Federal Jadyel Alencar estará no Fantástic...,Estaremos hoje no Fantástico para falar do pro...
2,ZzrG8BgHoZ0,As primeiras impressões sobre o ECA Digital | ...,E esse vai ser um dos assuntos da nossa coluna...
3,glr13drvkhE,"Além do tempo, o conteúdo também importa!",Deixa eu te mostrar o que seu filho pode estar...
4,pmmwmIJZ5_k,BASEUS WM01 É BOM? O MELHOR CUSTO-BENEFÍCIO EM...,O design dos fones de ouvido permite obter um ...
